# Module 3 • Classical Natural Language Processing

# Lesson 17 • Information Retrieval and Document Similarity

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Beginner to Intermediate  
**Estimated study time:** 100–130 minutes

---

## Scope

This lesson introduces classical Information Retrieval, inverted indexes,
Boolean search, TF-IDF ranking, cosine similarity, query-document matching,
top-k retrieval, retrieval evaluation, and multilingual search considerations.

## Learning Objectives

After completing this lesson, the learner should be able to:

- define Information Retrieval and distinguish it from text classification;
- explain document collections, queries, relevance, and rankings;
- build a simple inverted index;
- perform Boolean AND and OR retrieval;
- rank documents using TF-IDF and cosine similarity;
- explain why query and document vectors must share one vocabulary;
- calculate Precision@k, Recall@k, Average Precision, MRR, and nDCG;
- compare lexical overlap and vector similarity;
- identify vocabulary mismatch and zero-result queries;
- perform retrieval error analysis;
- explain the effect of preprocessing and tokenization on search;
- design basic English and Arabic retrieval pipelines.

## Table of Contents

1. What Is Information Retrieval?
2. Core Retrieval Concepts
3. Example Document Collection
4. Text Preparation for Retrieval
5. Inverted Indexes
6. Boolean Retrieval
7. Ranked Retrieval
8. TF-IDF Query-Document Matching
9. Cosine Similarity
10. Building a Search Function
11. Top-k Retrieval
12. Lexical Overlap Baseline
13. Query Expansion
14. Retrieval Evaluation
15. Precision@k and Recall@k
16. Mean Reciprocal Rank
17. Average Precision
18. nDCG
19. Error Analysis
20. Multilingual and Arabic Retrieval
21. Reproducibility and Deployment
22. Knowledge Check
23. Exercises
24. Summary and Next Lesson

# 1. What Is Information Retrieval?

**Information Retrieval (IR)** finds documents that are relevant to a user's
query.

Examples include:

- web search;
- digital-library search;
- enterprise document search;
- product search;
- legal search;
- support knowledge-base search;
- retrieval for question answering.

In [ ]:
import pandas as pd

ir_examples = pd.DataFrame(
    [
        ("Web search", "rank pages for a user query"),
        ("Knowledge base", "retrieve relevant support articles"),
        ("Digital library", "find papers by topic"),
        ("Enterprise search", "search reports and internal documents"),
        ("Question answering", "retrieve passages before answering"),
    ],
    columns=["Application", "Retrieval objective"],
)

ir_examples

Text classification assigns labels to documents. Retrieval ranks documents for
a query. A retrieval system may return no documents, one document, or many
ranked documents.

# 2. Core Retrieval Concepts

A retrieval system usually contains:

- **collection:** the searchable documents;
- **document identifier:** a stable ID for each item;
- **query:** the user's information need expressed in text;
- **index:** a structure that supports efficient matching;
- **retrieval model:** a method for scoring or selecting documents;
- **ranking:** documents ordered by estimated relevance;
- **relevance judgment:** whether a result satisfies the information need.

> **Key Idea**
>
> Retrieval quality depends on both matching and ranking. A relevant document
> that appears too low may still fail the user.

# 3. Example Document Collection

This notebook uses a small support knowledge base.

In [ ]:
documents = pd.DataFrame(
    [
        (
            "D1",
            "Password Reset",
            "Reset your password from the account security page. "
            "A verification code will be sent by email.",
        ),
        (
            "D2",
            "Invoice Download",
            "Download monthly invoices and payment receipts from "
            "the billing section.",
        ),
        (
            "D3",
            "Application Crash",
            "If the application crashes during startup, clear the "
            "cache and install the latest version.",
        ),
        (
            "D4",
            "Payment Declined",
            "A payment may be declined when the card has expired "
            "or the bank blocks the transaction.",
        ),
        (
            "D5",
            "Change Email Address",
            "Update the account email address after confirming "
            "your current password.",
        ),
        (
            "D6",
            "Slow Dashboard",
            "A slow dashboard may result from a weak connection, "
            "large reports, or browser extensions.",
        ),
        (
            "D7",
            "Subscription Cancellation",
            "Cancel a paid subscription from billing settings "
            "before the next renewal date.",
        ),
        (
            "D8",
            "Upload Failure",
            "File upload can fail because of unsupported formats, "
            "large files, or network interruption.",
        ),
    ],
    columns=["document_id", "title", "text"],
)

documents

In [ ]:
documents["search_text"] = (
    documents["title"]
    + ". "
    + documents["text"]
)

documents[["document_id", "search_text"]].head()

Titles are combined with document text so title words contribute to retrieval.
Production systems may assign different weights to title and body fields.

# 4. Text Preparation for Retrieval

Retrieval preprocessing may include:

- Unicode normalization;
- case normalization;
- tokenization;
- stop-word policy;
- stemming or lemmatization;
- structured-token handling;
- language-specific normalization.

Query and document preprocessing must remain consistent.

In [ ]:
import re

TOKEN_PATTERN = re.compile(r"\b\w+\b", flags=re.UNICODE)

def retrieval_tokens(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())


for text in documents["search_text"].head(2):
    print(retrieval_tokens(text))

Aggressive normalization may increase matching recall while reducing precision.
For example, stemming may connect `payments` with `payment`, but an overly
aggressive rule may merge unrelated words.

# 5. Inverted Indexes

An **inverted index** maps a term to the documents containing it.

Example:

```text
password → D1, D5
billing  → D2, D7
upload   → D8
```

In [ ]:
from collections import defaultdict

inverted_index = defaultdict(set)

for row in documents.itertuples(index=False):
    for token in set(retrieval_tokens(row.search_text)):
        inverted_index[token].add(row.document_id)

for term in ["password", "billing", "upload", "application"]:
    print(term, "->", sorted(inverted_index.get(term, set())))

Real search engines also store term frequency, positions, field information,
and document statistics.

# 6. Boolean Retrieval

Boolean retrieval selects documents using logical operators.

- `AND`: document must contain all query terms;
- `OR`: document may contain any query term;
- `NOT`: document must not contain a term.

In [ ]:
def boolean_and(query: str) -> list[str]:
    terms = retrieval_tokens(query)

    if not terms:
        return []

    postings = [
        inverted_index.get(term, set())
        for term in terms
    ]

    result = set.intersection(*postings) if postings else set()
    return sorted(result)


def boolean_or(query: str) -> list[str]:
    terms = retrieval_tokens(query)

    if not terms:
        return []

    result = set()

    for term in terms:
        result |= inverted_index.get(term, set())

    return sorted(result)


print("AND:", boolean_and("password email"))
print("OR: ", boolean_or("password email"))

Boolean retrieval is transparent but does not rank results by degree of
relevance.

# 7. Ranked Retrieval

Ranked retrieval assigns a score to each document and orders documents from
highest to lowest score.

A classical approach uses:

- TF-IDF vectors;
- cosine similarity.

Ranking is useful when many documents contain some query terms but differ in
how strongly they match the query.

# 8. TF-IDF Query-Document Matching

Documents and queries must be represented in the same feature space.

The vectorizer is fitted on the document collection. Queries are transformed
using that fitted vectorizer.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
)

document_matrix = tfidf_vectorizer.fit_transform(
    documents["search_text"]
)

print("Document matrix shape:", document_matrix.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

In [ ]:
query = "download payment invoice"
query_vector = tfidf_vectorizer.transform([query])

print("Query vector shape:", query_vector.shape)
print("Same number of features:", query_vector.shape[1] == document_matrix.shape[1])

Refitting the vectorizer on each query would produce an incompatible vocabulary
and invalid similarity scores.

# 9. Cosine Similarity

Cosine similarity compares vector directions.

For vectors \(a\) and \(b\):

\[
cosine(a,b) =
\frac{a \cdot b}{||a||\,||b||}
\]

Values closer to 1 indicate stronger vector similarity.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(
    query_vector,
    document_matrix,
).ravel()

ranking = documents[
    ["document_id", "title"]
].copy()

ranking["score"] = scores

ranking.sort_values(
    "score",
    ascending=False,
)

Cosine similarity measures weighted lexical overlap. It does not guarantee
semantic equivalence.

# 10. Building a Search Function

In [ ]:
def search_tfidf(
    query: str,
    top_k: int = 5,
) -> pd.DataFrame:
    if top_k <= 0:
        raise ValueError("top_k must be positive")

    vector = tfidf_vectorizer.transform([query])
    similarity_scores = cosine_similarity(
        vector,
        document_matrix,
    ).ravel()

    results = documents[
        ["document_id", "title", "text"]
    ].copy()

    results["score"] = similarity_scores

    return (
        results
        .sort_values(
            ["score", "document_id"],
            ascending=[False, True],
        )
        .head(top_k)
        .reset_index(drop=True)
    )


search_tfidf(
    "my card payment was rejected",
    top_k=4,
)

A minimum score threshold can prevent the system from returning documents with
no meaningful overlap.

In [ ]:
def search_with_threshold(
    query: str,
    top_k: int = 5,
    minimum_score: float = 0.05,
) -> pd.DataFrame:
    results = search_tfidf(
        query=query,
        top_k=len(documents),
    )

    filtered = results[
        results["score"] >= minimum_score
    ]

    return filtered.head(top_k).reset_index(drop=True)


search_with_threshold("weather forecast tomorrow")

Returning an empty result can be more honest than returning unrelated
documents.

# 11. Top-k Retrieval

`top_k` controls how many highest-ranked documents are returned.

A small value improves focus but may miss relevant results. A larger value may
improve recall while increasing user effort.

In [ ]:
for k in [1, 3, 5]:
    print(f"Top {k}")
    display(
        search_tfidf(
            "application does not start",
            top_k=k,
        )[["document_id", "title", "score"]]
    )

User-interface constraints and downstream use should influence the selected
value of `k`.

# 12. Lexical Overlap Baseline

A simple baseline ranks by the number of shared unique terms.

In [ ]:
def lexical_overlap_score(
    query: str,
    document: str,
) -> int:
    query_terms = set(retrieval_tokens(query))
    document_terms = set(retrieval_tokens(document))
    return len(query_terms & document_terms)


baseline_query = "reset account password"

baseline_results = documents[
    ["document_id", "title", "search_text"]
].copy()

baseline_results["overlap"] = baseline_results["search_text"].map(
    lambda text: lexical_overlap_score(
        baseline_query,
        text,
    )
)

baseline_results.sort_values(
    "overlap",
    ascending=False,
)[["document_id", "title", "overlap"]]

Baselines are important. A complex retrieval method should outperform a simple
matching strategy under relevant evaluation metrics.

# 13. Query Expansion

Query expansion adds related terms to improve matching.

Example:

```text
original: bill
expanded: bill invoice receipt payment
```

Expansion can increase recall but may reduce precision.

In [ ]:
expansion_dictionary = {
    "bill": ["invoice", "receipt", "payment"],
    "login": ["sign-in", "account", "credentials"],
    "crash": ["freeze", "error", "startup"],
}

def expand_query(query: str) -> str:
    tokens = retrieval_tokens(query)
    expanded = list(tokens)

    for token in tokens:
        expanded.extend(
            expansion_dictionary.get(token, [])
        )

    return " ".join(expanded)


original_query = "download bill"
expanded_query = expand_query(original_query)

print("Original:", original_query)
print("Expanded:", expanded_query)

In [ ]:
print("Original ranking:")
display(
    search_tfidf(
        original_query,
        top_k=3,
    )[["document_id", "title", "score"]]
)

print("Expanded ranking:")
display(
    search_tfidf(
        expanded_query,
        top_k=3,
    )[["document_id", "title", "score"]]
)

Expansion terms should be domain-appropriate and evaluated for query drift.

# 14. Retrieval Evaluation

Retrieval evaluation requires:

- a set of queries;
- relevance judgments;
- ranked results;
- suitable metrics.

Relevance may be binary or graded.

In [ ]:
relevance_judgments = {
    "reset password": {"D1", "D5"},
    "download invoice": {"D2"},
    "app crashes": {"D3"},
    "payment rejected": {"D4"},
    "cancel subscription": {"D7"},
}

relevance_judgments

Relevance judgments should reflect the user's information need, not only word
overlap.

# 15. Precision@k and Recall@k

**Precision@k** measures the fraction of the top-k results that are relevant.

**Recall@k** measures the fraction of all known relevant documents retrieved in
the top-k.

In [ ]:
def precision_at_k(
    ranked_ids: list[str],
    relevant_ids: set[str],
    k: int,
) -> float:
    retrieved = ranked_ids[:k]

    if not retrieved:
        return 0.0

    relevant_retrieved = sum(
        document_id in relevant_ids
        for document_id in retrieved
    )

    return relevant_retrieved / k


def recall_at_k(
    ranked_ids: list[str],
    relevant_ids: set[str],
    k: int,
) -> float:
    if not relevant_ids:
        return 0.0

    retrieved = ranked_ids[:k]
    relevant_retrieved = sum(
        document_id in relevant_ids
        for document_id in retrieved
    )

    return relevant_retrieved / len(relevant_ids)

In [ ]:
evaluation_query = "reset password"
ranked_ids = search_tfidf(
    evaluation_query,
    top_k=len(documents),
)["document_id"].tolist()

relevant_ids = relevance_judgments[evaluation_query]

for k in [1, 3, 5]:
    print(
        f"k={k}: "
        f"P@k={precision_at_k(ranked_ids, relevant_ids, k):.3f}, "
        f"R@k={recall_at_k(ranked_ids, relevant_ids, k):.3f}"
    )

Precision and recall express different priorities. Search interfaces often
prioritize early precision, while exhaustive legal or medical search may
prioritize recall.

# 16. Mean Reciprocal Rank

**Reciprocal Rank** is the inverse rank of the first relevant result.

```text
first relevant result at rank 1 → 1.0
first relevant result at rank 2 → 0.5
first relevant result at rank 4 → 0.25
```

**Mean Reciprocal Rank (MRR)** averages this value across queries.

In [ ]:
def reciprocal_rank(
    ranked_ids: list[str],
    relevant_ids: set[str],
) -> float:
    for rank, document_id in enumerate(
        ranked_ids,
        start=1,
    ):
        if document_id in relevant_ids:
            return 1.0 / rank

    return 0.0


reciprocal_rank(
    ranked_ids,
    relevant_ids,
)

In [ ]:
reciprocal_ranks = []

for query_text, relevant in relevance_judgments.items():
    ranked = search_tfidf(
        query_text,
        top_k=len(documents),
    )["document_id"].tolist()

    reciprocal_ranks.append(
        reciprocal_rank(ranked, relevant)
    )

mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

print("Reciprocal ranks:", reciprocal_ranks)
print(f"MRR: {mrr:.3f}")

MRR focuses only on the first relevant result.

# 17. Average Precision

**Average Precision (AP)** averages precision values at ranks where relevant
documents are retrieved.

In [ ]:
def average_precision(
    ranked_ids: list[str],
    relevant_ids: set[str],
) -> float:
    if not relevant_ids:
        return 0.0

    precision_values = []
    relevant_seen = 0

    for rank, document_id in enumerate(
        ranked_ids,
        start=1,
    ):
        if document_id in relevant_ids:
            relevant_seen += 1
            precision_values.append(
                relevant_seen / rank
            )

    return (
        sum(precision_values) / len(relevant_ids)
        if precision_values
        else 0.0
    )


ap_value = average_precision(
    ranked_ids,
    relevant_ids,
)

print(f"Average Precision: {ap_value:.3f}")

In [ ]:
average_precision_values = []

for query_text, relevant in relevance_judgments.items():
    ranked = search_tfidf(
        query_text,
        top_k=len(documents),
    )["document_id"].tolist()

    average_precision_values.append(
        average_precision(ranked, relevant)
    )

mean_average_precision = (
    sum(average_precision_values)
    / len(average_precision_values)
)

print("AP values:", average_precision_values)
print(f"MAP: {mean_average_precision:.3f}")

Mean Average Precision considers the ranks of multiple relevant documents.

# 18. nDCG

**Normalized Discounted Cumulative Gain (nDCG)** supports graded relevance.

Highly relevant documents receive greater gain, while lower ranks receive a
discount.

In [ ]:
import math

def dcg_at_k(
    relevance_scores: list[float],
    k: int,
) -> float:
    total = 0.0

    for index, relevance in enumerate(
        relevance_scores[:k],
        start=1,
    ):
        total += (
            (2 ** relevance - 1)
            / math.log2(index + 1)
        )

    return total


def ndcg_at_k(
    relevance_scores: list[float],
    k: int,
) -> float:
    actual_dcg = dcg_at_k(
        relevance_scores,
        k,
    )

    ideal_scores = sorted(
        relevance_scores,
        reverse=True,
    )

    ideal_dcg = dcg_at_k(
        ideal_scores,
        k,
    )

    return (
        actual_dcg / ideal_dcg
        if ideal_dcg > 0
        else 0.0
    )


graded_relevance = [3, 0, 2, 1, 0]

print(f"nDCG@5: {ndcg_at_k(graded_relevance, 5):.3f}")

nDCG is useful when relevance is not simply relevant or irrelevant.

# 19. Error Analysis

Retrieval errors may include:

- relevant document missing from top-k;
- irrelevant document ranked too high;
- zero-result query;
- vocabulary mismatch;
- incorrect tokenization;
- overly aggressive stop-word removal;
- query expansion drift;
- duplicate results;
- stale index;
- language mismatch.

In [ ]:
analysis_queries = [
    "forgot credentials",
    "billing receipt",
    "program freezes",
    "weather forecast",
]

error_analysis_rows = []

for query_text in analysis_queries:
    top_result = search_tfidf(
        query_text,
        top_k=1,
    ).iloc[0]

    error_analysis_rows.append(
        {
            "query": query_text,
            "top_document": top_result["document_id"],
            "top_title": top_result["title"],
            "score": top_result["score"],
        }
    )

pd.DataFrame(error_analysis_rows)

Queries with zero or very low scores should be reviewed separately from
incorrectly ranked queries.

## 19.1 Vocabulary Mismatch

Lexical retrieval may fail when the query and document use different words:

```text
query: forgot credentials
document: reset password
```

Possible responses include:

- stemming or lemmatization;
- synonyms;
- query expansion;
- character n-grams;
- semantic retrieval;
- hybrid retrieval.

# 20. Multilingual and Arabic Retrieval

Multilingual retrieval must consider:

- language identification;
- script and Unicode;
- tokenization;
- morphology;
- diacritics;
- spelling variation;
- dialect;
- translation;
- code-switching.

## 20.1 Arabic Retrieval

Arabic word forms may contain attached clitics and rich morphology.

Example variants:

```text
كتاب
الكتاب
والكتاب
كتابه
```

Word-level matching may treat them as separate features.

In [ ]:
arabic_documents = pd.DataFrame(
    [
        ("A1", "إعادة تعيين كلمة المرور"),
        ("A2", "تنزيل الفاتورة وإيصال الدفع"),
        ("A3", "التطبيق يتوقف عند التشغيل"),
        ("A4", "إلغاء الاشتراك المدفوع"),
    ],
    columns=["document_id", "text"],
)

arabic_word_vectorizer = TfidfVectorizer(
    analyzer="word",
)

arabic_char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
)

arabic_word_matrix = arabic_word_vectorizer.fit_transform(
    arabic_documents["text"]
)

arabic_char_matrix = arabic_char_vectorizer.fit_transform(
    arabic_documents["text"]
)

print("Word matrix shape:", arabic_word_matrix.shape)
print("Character matrix shape:", arabic_char_matrix.shape)

In [ ]:
arabic_query = "أريد تحميل الفاتورة"

word_scores = cosine_similarity(
    arabic_word_vectorizer.transform([arabic_query]),
    arabic_word_matrix,
).ravel()

char_scores = cosine_similarity(
    arabic_char_vectorizer.transform([arabic_query]),
    arabic_char_matrix,
).ravel()

arabic_comparison = arabic_documents.copy()
arabic_comparison["word_score"] = word_scores
arabic_comparison["character_score"] = char_scores

arabic_comparison.sort_values(
    "character_score",
    ascending=False,
)

Character n-grams may improve robustness to variation, but they can also match
unrelated strings. Evaluation remains necessary.

# 21. Reproducibility and Deployment

Record:

- collection version;
- document IDs;
- preprocessing configuration;
- vectorizer vocabulary;
- IDF values;
- query-expansion resources;
- relevance judgments;
- metric definitions;
- software versions.

Production monitoring should track:

- query frequency;
- zero-result rate;
- click or selection behavior;
- stale documents;
- latency;
- index failures;
- language distribution;
- drift in query vocabulary.

In [ ]:
import sklearn

retrieval_metadata = pd.Series(
    {
        "documents": len(documents),
        "vectorizer": "word TF-IDF, 1-2 grams",
        "similarity": "cosine",
        "scikit-learn_version": sklearn.__version__,
        "relevance_queries": len(relevance_judgments),
    },
    name="Retrieval experiment",
)

retrieval_metadata

# 22. Knowledge Check

1. What is Information Retrieval?
2. How does retrieval differ from classification?
3. What is an inverted index?
4. How do Boolean AND and OR differ?
5. Why is ranked retrieval useful?
6. Why must queries use the fitted document vectorizer?
7. What does cosine similarity measure?
8. What does `top_k` control?
9. Why can a score threshold be useful?
10. What is query expansion?
11. How do Precision@k and Recall@k differ?
12. What does MRR emphasize?
13. What does Average Precision measure?
14. Why is nDCG useful for graded relevance?
15. Which Arabic characteristics affect retrieval?

# 23. Exercises

## Exercise 1 — Inverted Index

Build an inverted index that stores document IDs and term frequencies.

## Exercise 2 — Boolean Search

Add `NOT` and parenthesized expressions to the Boolean retrieval functions.

## Exercise 3 — Ranked Search

Build a TF-IDF search engine for a new document collection.

## Exercise 4 — Field Weighting

Give title terms more weight than body terms.

## Exercise 5 — Query Expansion

Create a small domain synonym dictionary and measure its effect on recall.

## Exercise 6 — Retrieval Metrics

Calculate Precision@k, Recall@k, MRR, MAP, and nDCG for at least ten queries.

## Exercise 7 — Error Analysis

Group failures into zero-result, vocabulary mismatch, incorrect ranking, and
irrelevant expansion.

## Exercise 8 — Arabic Retrieval

Compare Arabic word TF-IDF, character TF-IDF, and normalized word TF-IDF.

## Challenge Exercises

1. Implement BM25 from scratch.
2. Create a hybrid Boolean and TF-IDF search engine.
3. Add phrase matching using token positions.
4. Build a multilingual search index.
5. Add a simple relevance-feedback mechanism.

# 24. Summary and Next Lesson

In this lesson:

- Information Retrieval ranked documents for user queries;
- inverted indexes mapped terms to documents;
- Boolean retrieval supported exact logical matching;
- TF-IDF and cosine similarity supported ranked retrieval;
- query and document vectors shared one fitted feature space;
- top-k and score thresholds controlled result presentation;
- lexical overlap provided a useful baseline;
- query expansion improved recall but risked query drift;
- Precision@k and Recall@k measured early retrieval quality;
- MRR emphasized the first relevant result;
- Average Precision and MAP considered multiple relevant results;
- nDCG supported graded relevance;
- retrieval error analysis exposed vocabulary mismatch and zero-result queries;
- Arabic retrieval required attention to morphology, clitics, and spelling
  variation.

## Next Lesson

**Lesson 18: Topic Modeling with Latent Dirichlet Allocation** introduces
unsupervised topic discovery, document-topic distributions, topic-word
distributions, preprocessing choices, interpretation, and topic-model
evaluation.

# References

- Manning, C. D., Raghavan, P., & Schütze, H. *Introduction to Information Retrieval*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- scikit-learn TF-IDF and cosine-similarity documentation.
- classical Information Retrieval evaluation literature.
- Arabic Information Retrieval literature.